# Hari 23 — Iterasi Data Preparation: Menambah Fitur Lag-0

**Recap keputusan Hari 22:** proyek diputar balik ke Data Preparation untuk menambah fitur `lag_0` — kasus minggu ini, yang sebenarnya SUDAH diketahui saat kita membuat prediksi untuk minggu depan, tapi selama ini tidak pernah dimasukkan sebagai fitur (fitur paling baru yang ada cuma `lag_1`, mundur 1 minggu).

Hari ini: bangun fitur ini, cek dampaknya ke multikolinearitas, lalu siapkan ulang seluruh file train/test dengan versi baru (diberi akhiran `_v2` supaya tidak menimpa file lama — file lama tetap berguna sebagai pembanding "sebelum vs sesudah").

In [25]:
# Cell ini sudah lengkap.
import pandas as pd
#import numpy as np
#import matplotlib.pyplot as plt

df_fitur = pd.read_csv("dataset_siap_modeling.csv", index_col=0, parse_dates=True)
kasus_asli = pd.read_csv("dataset_bersih_minggu2.csv", index_col=0, parse_dates=True)["kasus_baru_mingguan"]

print(f"df_fitur (versi lama): {df_fitur.shape[0]} baris, kolom: {list(df_fitur.columns)}")

df_fitur (versi lama): 143 baris, kolom: ['lag_1', 'lag_2', 'lag_3', 'rolling_mean_4w', 'vaksin_persen', 'bulan_1', 'bulan_2', 'bulan_3', 'bulan_4', 'bulan_5', 'bulan_6', 'bulan_7', 'bulan_8', 'bulan_9', 'bulan_10', 'bulan_11', 'bulan_12', 'target_minggu_depan']


## Membangun Fitur Lag-0

`lag_0` itu beda dari `lag_1`, `lag_2`, `lag_3` — dia **tidak perlu di-shift sama sekali**. Untuk baris pada tanggal d, `lag_0` = kasus_baru_mingguan pada tanggal d itu sendiri (nilai yang sudah kita ketahui saat berdiri di titik waktu itu, sebelum memprediksi minggu depan).

In [26]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Buat df_fitur_v2 = df_fitur.copy() (supaya tidak mengubah df_fitur asli)
# 2. Buat kolom baru "lag_0" di df_fitur_v2, isinya kasus_asli yang di-align ke
#    index df_fitur_v2 (pakai .loc[df_fitur_v2.index], SAMA seperti current_actual_test
#    di Hari 18 — bedanya sekarang untuk SELURUH baris, bukan cuma test set)
# 3. Cek tidak ada NaN di kolom lag_0 baru ini: df_fitur_v2["lag_0"].isnull().sum()
#    (harusnya 0, karena lag_0 tidak butuh histori sebelumnya seperti lag_1/2/3)
# 4. Cetak df_fitur_v2[["lag_0", "lag_1", "lag_2", "lag_3"]].head() untuk melihat
#    polanya: lag_0 tanggal ini harusnya sama dengan lag_1 tanggal MINGGU DEPANNYA
# Tulis kode kamu di bawah ini:
df_fitur_v2 = df_fitur.copy()
df_fitur_v2["lag_0"] = kasus_asli.loc[df_fitur_v2.index]
print(f"Jumalah Nan: {df_fitur_v2["lag_0"].isnull().sum()}")
print(df_fitur_v2[["lag_0", "lag_1", "lag_2", "lag_3"]].head())

Jumalah Nan: 0
                            lag_0   lag_1   lag_2  lag_3
Date                                                    
2020-03-29 00:00:00+00:00   771.0   397.0   111.0    6.0
2020-04-05 00:00:00+00:00   988.0   771.0   397.0  111.0
2020-04-12 00:00:00+00:00  1968.0   988.0   771.0  397.0
2020-04-19 00:00:00+00:00  2334.0  1968.0   988.0  771.0
2020-04-26 00:00:00+00:00  2307.0  2334.0  1968.0  988.0


## Cek Dampak ke Multikolinearitas

Dugaan kuat: `lag_0` akan makin memperparah korelasi antar fitur lag (karena secara alami kasus minggu ini SANGAT dekat dengan kasus minggu lalu). Ini tidak berarti fiturnya buruk — cuma berarti interpretasi koefisien individual makin perlu hati-hati (seperti sudah kita bahas di Hari 16), sementara model berbasis pohon (Random Forest) tetap relatif aman.

In [27]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Hitung correlation matrix dari df_fitur_v2[["lag_0", "lag_1", "lag_2", "lag_3",
#    "rolling_mean_4w"]] pakai .corr()
# 2. Cetak hasilnya
# 3. Bandingkan angka korelasi lag_0 terhadap lag_1 dengan angka korelasi lag_1
#    terhadap lag_2 dari Hari 16 (0,76-0,98) — apakah korelasi lag_0 lebih tinggi lagi?
# Tulis kode kamu di bawah ini:
df_fitur_v2[["lag_0", "lag_1", "lag_2", "lag_3"]].corr()

korelasi_lag = df_fitur_v2[["lag_0", "lag_1", "lag_2", "lag_3"]].corr()
print(f"Korelasi lag_0 dengan lag_1: {korelasi_lag.loc["lag_0", "lag_1"]:.4f}")
print(f"Korelasi lag_1 dengan lag_2: {korelasi_lag.loc["lag_1", "lag_2"]:.4f}")

Korelasi lag_0 dengan lag_1: 0.9279
Korelasi lag_1 dengan lag_2: 0.9280


In [ ]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Hitung correlation matrix dari df_fitur_v2[["lag_0", "lag_1", "lag_2", "lag_3",
#    "rolling_mean_4w"]] pakai .corr()
# 2. Cetak hasilnya
# 3. Bandingkan angka korelasi lag_0 terhadap lag_1 dengan angka korelasi lag_1
#    terhadap lag_2 dari Hari 16 (0,76-0,98) — apakah korelasi lag_0 lebih tinggi lagi?
# Tulis kode kamu di bawah ini:
"""korelasi_lag = df_fitur_v2[["lag_0", "lag_1", "lag_2", "lag_3", "rolling_mean_4w"]].corr()
print(korelasi_lag)

print(f"Korelasi lag_0 vs lag_1: {korelasi_lag.loc['lag_0', 'lag_1']:.4f}")
print(f"Korelasi lag_1 vs lag_2: {korelasi_lag.loc['lag_1', 'lag_2']:.4f}")"""

                    lag_0     lag_1     lag_2     lag_3  rolling_mean_4w
lag_0            1.000000  0.927869  0.765695  0.562770         0.877736
lag_1            0.927869  1.000000  0.928017  0.766368         0.976497
lag_2            0.765695  0.928017  1.000000  0.928260         0.976557
lag_3            0.562770  0.766368  0.928260  1.000000         0.878383
rolling_mean_4w  0.877736  0.976497  0.976557  0.878383         1.000000
Korelasi lag_0 vs lag_1: 0.9279
Korelasi lag_1 vs lag_2: 0.9280


## Menyiapkan Ulang X_full, Train-Test Split, dan Scaling (Versi 2)

In [29]:
# Cell ini sudah lengkap — susun ulang X_full/y_full dengan fitur lag_0 ditambahkan,
# lalu simpan sebagai file versi baru (bukan menimpa file lama).

X_full_v2 = df_fitur_v2.drop(columns=["target_minggu_depan"])
y_full_v2 = df_fitur_v2["target_minggu_depan"]

final_v2 = pd.concat([X_full_v2, y_full_v2], axis=1)
final_v2.to_csv("dataset_siap_modeling_v2.csv")
print(f"Tersimpan: dataset_siap_modeling_v2.csv — {X_full_v2.shape[1]} fitur (sebelumnya {df_fitur.shape[1] - 1})")

Tersimpan: dataset_siap_modeling_v2.csv — 18 fitur (sebelumnya 17)


In [31]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# (ini pengulangan pola Hari 13-14, dengan fitur baru — cara ini melatih kamu memastikan
# proses yang sama bisa diulang konsisten kalau data/fitur berubah, bukan cuma sekali pakai)

# 1. Tentukan cutoff = int(len(X_full_v2) * 0.8) — SAMA seperti Hari 13
# 2. Split berurutan waktu: X_train_v2 = X_full_v2.iloc[:cutoff], X_test_v2 = sisanya
#    (begitu juga y_train_v2, y_test_v2)
# 3. Fit StandardScaler HANYA dari X_train_v2 (kolom kontinu saja: lag_0, lag_1, lag_2,
#    lag_3, rolling_mean_4w, vaksin_persen — kolom bulan_* tetap tidak di-scale)
# 4. Transform X_train_v2 dan X_test_v2 kolom kontinunya, gabungkan lagi dengan kolom
#    bulan_* seperti pola Hari 13, simpan sebagai X_train_scaled_v2, X_test_scaled_v2
# 5. Simpan keenam file (X_train_v2, X_test_v2, X_train_scaled_v2, X_test_scaled_v2,
#    y_train_v2, y_test_v2) ke CSV dengan akhiran _v2
# 6. Assert X_train_v2.index.max() < X_test_v2.index.min() — sanity check wajib seperti Hari 14
# Tulis kode kamu di bawah ini:
from sklearn.preprocessing import StandardScaler
cutoff = int(len(X_full_v2)* 0.8)
X_train_v2 = X_full_v2.iloc[:cutoff]
X_test_v2 = X_full_v2.iloc[cutoff:]
y_train_v2 = y_full_v2.iloc[:cutoff]
y_test_v2 = y_full_v2.iloc[cutoff:]

scaler = StandardScaler()
kolom_kontinu = ["lag_0", "lag_1", "lag_2", "lag_3", "rolling_mean_4w", "vaksin_persen"]
kolom_bulan = [col for col in X_full_v2.columns if col.startswith("bulan_")]

X_train_scaled_kontinu_v2 = scaler.fit_transform(X_train_v2[kolom_kontinu])
X_test_scaled_kontinu_v2 = scaler.transform(X_test_v2[kolom_kontinu])

X_train_scaled_v2 = pd.DataFrame(X_train_scaled_kontinu_v2, columns=kolom_kontinu, index=X_train_v2.index)
X_train_scaled_v2 = pd.concat([X_train_scaled_v2, X_train_v2[kolom_bulan]], axis=1)

X_test_scaled_v2 = pd.DataFrame(X_test_scaled_kontinu_v2, columns=kolom_kontinu, index=X_test_v2.index)
X_test_scaled_v2 = pd.concat([X_test_scaled_v2, X_test_v2[kolom_bulan]], axis=1)

X_train_v2.to_csv("X_train_v2.csv")
X_test_v2.to_csv("X_test_v2.csv")
X_train_scaled_v2.to_csv("X_train_scaled_v2.csv")
X_test_scaled_v2.to_csv("X_test_scaled_v2.csv")
y_train_v2.to_csv("y_train_v2.csv")
y_test_v2.to_csv("y_test_v2.csv")
assert X_train_v2.index.max() < X_test_v2.index.min()

## Refleksi Hari 23

> 1. Berapa korelasi `lag_0` terhadap `lag_1`? Apakah lebih tinggi dari korelasi antar fitur lag lain yang sudah kamu temukan di Hari 16? → **0.9279**
> 2. Berapa jumlah fitur sekarang dibanding sebelumnya (lihat output cell "Tersimpan")? → 18 sembelumnya 17
> 3. Dengan multikolinearitas yang kemungkinan makin parah, model mana yang menurutmu PALING BERISIKO terpengaruh — Linear Regression atau Random Forest? Kenapa? → **Belum tau, harus diuji dulu**

---
### Selanjutnya: Hari 24 — Retrain & Bandingkan

Besok kita latih ulang Linear Regression dan Random Forest pakai `X_train_v2`/`X_test_v2` (dengan `lag_0`), lalu bandingkan MAE single-split-nya langsung dengan angka lama (Linear: 7.806,56 | RF tuned: 8.456,32) — apakah fitur baru ini benar-benar membantu, sesuai dugaan dari Hari 18?